# Module 12: Context Engineering

**Goal:** Design what enters the context window — not just fill it.

**What you'll learn:**
- Token budget management and per-slot allocation
- Observation masking — compress tool outputs before they enter context
- Provider-level prefix caching (automatic + explicit breakpoints)
- Context compression strategies

**Prerequisites:** Module 00 (tokens), Module 06 (caching basics)

---

## 0. Setup

In [ ]:
%pip install -q tiktoken

In [ ]:
import tiktoken
from dataclasses import dataclass, field

enc = tiktoken.get_encoding("cl100k_base")
print("✅ Setup complete")

---
## 1. Token Budget Tracker

Enforces hard per-slot token limits. Prevents any single component from crowding out others.

In [ ]:
class TokenBudget:
    def __init__(self, total: int):
        self.total = total
        self._enc = tiktoken.get_encoding("cl100k_base")
        self._alloc = {}
        self._used = {}

    def allocate(self, slot: str, tokens: int):
        committed = sum(self._alloc.values())
        if committed + tokens > self.total:
            raise ValueError(f"Only {self.total - committed} tokens remaining")
        self._alloc[slot] = tokens
        self._used[slot] = 0

    def consume(self, slot: str, text: str) -> str:
        limit = self._alloc[slot]
        tokens = self._enc.encode(text)
        if len(tokens) > limit:
            tokens = tokens[:limit]
            text = self._enc.decode(tokens)
        self._used[slot] = len(tokens)
        return text

    def report(self):
        print(f"\nToken Budget (total: {self.total:,})")
        for slot, alloc in self._alloc.items():
            used = self._used.get(slot, 0)
            print(f"  {slot:<20} {used:>5}/{alloc:<5} ({used/alloc*100:.0f}%)")
        print(f"  {'REMAINING':<20} {self.total - sum(self._used.values()):>5}")

# Demo
budget = TokenBudget(total=4000)
budget.allocate("system_prompt", 400)
budget.allocate("retrieved_docs", 1800)
budget.allocate("conversation", 1200)
budget.allocate("response", 600)

budget.consume("system_prompt", "You are an expert analyst. Be precise and quantitative.")
budget.consume("retrieved_docs", "Annual report: " + "Revenue grew 24% YoY. " * 100)
budget.consume("conversation", "User: What was the growth rate?\nAssistant: 24% YoY.")

budget.report()

---
## 2. Observation Masking

Compress tool outputs before they enter context. A web search returning 45,000 tokens to answer "What's the homepage title?" wastes context.

In [ ]:
import re

class ObservationMasker:
    def __init__(self, max_tokens: int = 500):
        self.max_tokens = max_tokens
        self._enc = tiktoken.get_encoding("cl100k_base")

    def mask(self, raw_output: str) -> str:
        tokens = self._enc.encode(raw_output)
        if len(tokens) <= self.max_tokens:
            return raw_output
        # Structural extraction: strip HTML, collapse whitespace
        text = re.sub(r'<[^>]+>', '', raw_output)
        text = re.sub(r'\n{3,}', '\n\n', text).strip()
        # Truncate to budget
        tokens = self._enc.encode(text)[:self.max_tokens]
        return self._enc.decode(tokens)

# Demo: large HTML page → compressed output
big_html = "<html><body>" + "<p>Sentence about the topic. </p>" * 500 + "</body></html>"
masker = ObservationMasker(max_tokens=100)
compressed = masker.mask(big_html)

original_tokens = len(enc.encode(big_html))
compressed_tokens = len(enc.encode(compressed))
print(f"Original: {original_tokens:,} tokens")
print(f"Compressed: {compressed_tokens} tokens")
print(f"Reduction: {(1 - compressed_tokens/original_tokens)*100:.1f}%")
print(f"Preview: {compressed[:200]}...")

---
## 3. Sliding Window Compression

Keep first N turns (establish context) + last M turns (recent), drop the middle.

In [ ]:
def sliding_window_compress(messages, anchor_turns=2, recent_turns=6):
    if len(messages) <= (anchor_turns + recent_turns) * 2:
        return messages
    anchors = messages[:anchor_turns * 2]
    recent = messages[-(recent_turns * 2):]
    dropped = len(messages) - len(anchors) - len(recent)
    summary = {"role": "system", "content": f"[{dropped} earlier messages omitted]"}
    return anchors + [summary] + recent

# Simulate a long conversation
messages = [
    {"role": "user", "content": f"Turn {i}: Tell me about topic {i}"}
    for i in range(20)
]

print(f"Original: {len(messages)} messages")
compressed = sliding_window_compress(messages, anchor_turns=2, recent_turns=4)
print(f"Compressed: {len(compressed)} messages")
for m in compressed:
    print(f"  {m['role']}: {m['content'][:60]}")

---
## Summary

| Technique | When to Use |
|-----------|-------------|
| **Token Budget** | Multi-component context (docs + history + tools) |
| **Observation Masking** | Large tool outputs (HTML, search results, API responses) |
| **Sliding Window** | Long conversations that exceed context limits |
| **Prefix Caching** | Stable system prompts reused across requests |

**Core insight:** Your context window is not a buffer to fill — it's a signal to design.

---
## 🧪 Exercises

1. **Budget Enforcement**: Build a `TokenBudget` with 4,000 total tokens. Allocate 400 to system, 1,600 to docs, 1,200 to history, 800 to response. Feed it increasingly long documents and verify truncation.

2. **Masking Benchmark**: Apply the `ObservationMasker` at `max_tokens=100`, `300`, `500`. Measure output reduction and whether compressed output still answers the question.

3. **Window Comparison**: Compare sliding window compression with LLM-based summarization. Which preserves more information?